# 06.2 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/02-kmeans-by-hand/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [2]:
names = list("ABCDEFGH")
points = np.array([[1, 1], [1, 3], [2, 2], [4, 3],      # x = orders last year
                   [6, 6], [7, 5], [7, 8], [9, 7]],     # y = average items per order
                  dtype=float)
pd.DataFrame(points, index=names, columns=["orders", "items"]).T

,A,B,C,D,E,F,G,H
orders,1.0,1.0,2.0,4.0,6.0,7.0,7.0,9.0
items,1.0,3.0,2.0,3.0,6.0,5.0,8.0,7.0


In [3]:
def kmeans_by_hand(points, centroids, max_iter=10):
    """Plain K-Means: assign to nearest centroid, move centroids to the mean, repeat."""
    labels = None
    for it in range(1, max_iter + 1):
        dist = np.sqrt(((points[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2))
        new_labels = dist.argmin(axis=1)                         # assignment step
        inertia = ((points - centroids[new_labels]) ** 2).sum()
        yield it, centroids.copy(), dist, new_labels, inertia
        if labels is not None and (new_labels == labels).all():
            return                                               # nothing changed: converged
        labels = new_labels
        centroids = np.array([points[labels == j].mean(axis=0)   # update step
                              for j in range(len(centroids))])

## Exercise 1

Iteration 1 with centroids G (7, 8) and H (9, 7): only F and H are nearer to H; everyone else, including far-away A, joins G's team. New centroids: mean of A, B, C, D, E, G = (3.5, 3.83) and mean of F, H = (8, 6).

Iteration 2: A, B, C, D join the first centroid and E, F, G, H the second. New centroids (2, 2.25) and (7.25, 6.5), the same as in the lesson.

Iteration 3: nothing changes. **Same final clusters, also three iterations.** On these eight well-separated customers the start does not matter; on harder data it does (lesson section 4).

In [4]:
for it, cents, dist, labels, inertia in kmeans_by_hand(points, points[[6, 7]]):
    print(f"iteration {it}: centroids {cents.round(2).tolist()}  clusters {labels + 1}  inertia {inertia:.2f}")

iteration 1: centroids [[7.0, 8.0], [9.0, 7.0]]  clusters [1 1 1 1 1 2 1 2]  inertia 254.00
iteration 2: centroids [[3.5, 3.83], [8.0, 6.0]]  clusters [1 1 1 1 2 2 2 2]  inertia 40.78
iteration 3: centroids [[2.0, 2.25], [7.25, 6.5]]  clusters [1 1 1 1 2 2 2 2]  inertia 18.50


## Exercise 2

Centroids: {A, B, C} → (1.33, 2); {D, E, F, G, H} → (6.6, 5.8).

- A: $(1-1.33)^2 + (1-2)^2 = 0.111 + 1 = 1.111$; B: 1.111; C: $0.444 + 0 = 0.444$. Cluster 1: **2.667**.
- D: $(4-6.6)^2 + (3-5.8)^2 = 6.76 + 7.84 = 14.6$; E: 0.4; F: 0.8; G: 5.0; H: 7.2. Cluster 2: **28.0**.

Total **30.67**, much higher than 18.5. Moving D into the second group makes that group far less tight. K-Means found the better split because it keeps lowering inertia.

In [5]:
labels = np.array([0, 0, 0, 1, 1, 1, 1, 1])
cents = np.array([points[labels == j].mean(axis=0) for j in range(2)])
print("centroids:", cents.round(2).tolist())
print("inertia:  ", round(((points - cents[labels]) ** 2).sum(), 3))

centroids: [[1.33, 2.0], [6.6, 5.8]]
inertia:   30.667


## Exercise 3

In [6]:
from sklearn.cluster import KMeans
for k in range(1, 9):
    print(f"k = {k}: inertia {KMeans(n_clusters=k, n_init=10, random_state=42).fit(points).inertia_:.2f}")

k = 1: inertia 109.75
k = 2: inertia 18.50
k = 3: inertia 12.25
k = 4: inertia 6.17
k = 5: inertia 3.67
k = 6: inertia 2.00
k = 7: inertia 1.00
k = 8: inertia 0.00


Inertia always falls as k grows, and at **k = 8 it is 0**: every customer is its own cluster and sits exactly on its centroid. So "lowest inertia" cannot be how you choose k. Lesson 06.3 looks for the point where adding clusters stops helping much (the elbow).

## Exercise 4

In [7]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)    # 2011-12-10
rfm = sales.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"))
print(f"{len(sales):,} rows, {len(rfm):,} customers, snapshot {snapshot.date()}")

392,693 rows, 4,338 customers, snapshot 2011-12-10


In [8]:
from sklearn.preprocessing import StandardScaler
X_nolog = StandardScaler().fit_transform(rfm)
labels = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_nolog)
print("sizes, scaled without log:", np.sort(np.bincount(labels))[::-1])
rfm.assign(cluster=labels).groupby("cluster").mean().round(0)

sizes, scaled without log: [3060 1061  204   13]


,recency,frequency,monetary
cluster,,,
0,43.0,4.0,1353.0
1,248.0,2.0,476.0
2,7.0,83.0,127188.0
3,15.0,22.0,12691.0


Better than raw numbers (recency now counts), but the long tails of frequency and monetary still stretch the scale: a few extreme customers get small clusters of their own, and most customers are squeezed into two big groups split mainly by recency. Scaling fixes units; **the log fixes skew**. You need both.